In [1]:
import pandas as pd

# Your existing per-borough SARIMA forecasts, July-Dec 2026
sarima_forecast = pd.read_csv("borough_future_forecasts_Jul-Dec_2026.csv")
print(sarima_forecast.shape)
print(sarima_forecast.head())

# Full historical crime data (needed to pull 2025 actuals for naive seasonal baseline)
crime_full = pd.read_csv("data/cleaned_crime_long_through_june2026.csv")
crime_full["Date"] = pd.to_datetime(crime_full["Date"])
print(crime_full.columns.tolist())

(192, 5)
                Borough    Month  Forecast  Lower_95CI  Upper_95CI
0  Barking and Dagenham  2026-07      1930        1709        2152
1  Barking and Dagenham  2026-08      1810        1536        2085
2  Barking and Dagenham  2026-09      1872        1545        2200
3  Barking and Dagenham  2026-10      1975        1604        2346
4  Barking and Dagenham  2026-11      1968        1558        2378
['Borough', 'MajorCategory', 'MinorCategory', 'Date', 'Year', 'Month', 'CrimeCount']


In [2]:
# Build a mapping from 2026 target months to their 2025 equivalents
month_map = {
    "2026-07": "2025-07-01", "2026-08": "2025-08-01", "2026-09": "2025-09-01",
    "2026-10": "2025-10-01", "2026-11": "2025-11-01", "2026-12": "2025-12-01"
}

# Get per-borough totals for those 2025 months
naive_dates = pd.to_datetime(list(month_map.values()))
borough_naive_raw = crime_full[crime_full["Date"].isin(naive_dates)]
borough_naive_raw = borough_naive_raw.groupby(["Borough", "Date"])["CrimeCount"].sum().reset_index()

# Map each 2025 date back to its corresponding 2026 month label
date_to_month2026 = {v: k for k, v in month_map.items()}
borough_naive_raw["Month"] = borough_naive_raw["Date"].dt.strftime("%Y-%m-%d").map(date_to_month2026)
borough_naive = borough_naive_raw[["Borough", "Month", "CrimeCount"]].rename(columns={"CrimeCount": "Naive_Seasonal"})

print(borough_naive.shape)
print(borough_naive.head(8))

(192, 3)
                Borough    Month  Naive_Seasonal
0  Barking and Dagenham  2026-07            1792
1  Barking and Dagenham  2026-08            1635
2  Barking and Dagenham  2026-09            1646
3  Barking and Dagenham  2026-10            1697
4  Barking and Dagenham  2026-11            1739
5  Barking and Dagenham  2026-12            1622
6                Barnet  2026-07            2587
7                Barnet  2026-08            2314


In [3]:
# Merge SARIMA forecast with naive seasonal baseline
ensemble_df = sarima_forecast.merge(borough_naive, on=["Borough", "Month"])

# Apply the established ensemble weights: 0.4 SARIMA / 0.6 Naive Seasonal
ensemble_df["Ensemble_Forecast"] = (0.4 * ensemble_df["Forecast"]) + (0.6 * ensemble_df["Naive_Seasonal"])
ensemble_df["Ensemble_Forecast"] = ensemble_df["Ensemble_Forecast"].round(0)

print(ensemble_df.shape)
print(ensemble_df[["Borough", "Month", "Forecast", "Naive_Seasonal", "Ensemble_Forecast"]].head(10))

# Save it
ensemble_df.to_csv("data/borough_ensemble_forecast_jul_dec_2026.csv", index=False)
print("\nSaved borough_ensemble_forecast_jul_dec_2026.csv")

(192, 7)
                Borough    Month  Forecast  Naive_Seasonal  Ensemble_Forecast
0  Barking and Dagenham  2026-07      1930            1792             1847.0
1  Barking and Dagenham  2026-08      1810            1635             1705.0
2  Barking and Dagenham  2026-09      1872            1646             1736.0
3  Barking and Dagenham  2026-10      1975            1697             1808.0
4  Barking and Dagenham  2026-11      1968            1739             1831.0
5  Barking and Dagenham  2026-12      1781            1622             1686.0
6                Barnet  2026-07      2807            2587             2675.0
7                Barnet  2026-08      2549            2314             2408.0
8                Barnet  2026-09      2627            2348             2460.0
9                Barnet  2026-10      2783            2528             2630.0

Saved borough_ensemble_forecast_jul_dec_2026.csv


In [4]:
london_ensemble = ensemble_df.groupby("Month")[["Forecast", "Naive_Seasonal", "Ensemble_Forecast"]].sum().reset_index()
print("\nLondon-wide total, July-Dec 2026:")
print(london_ensemble)

london_ensemble.to_csv("data/london_ensemble_forecast_jul_dec_2026.csv", index=False)
print("Saved london_ensemble_forecast_jul_dec_2026.csv")


London-wide total, July-Dec 2026:
     Month  Forecast  Naive_Seasonal  Ensemble_Forecast
0  2026-07     84666           82509            83371.0
1  2026-08     81433           77380            79003.0
2  2026-09     78226           73857            75605.0
3  2026-10     83051           76963            79396.0
4  2026-11     81061           76013            78033.0
5  2026-12     77473           73393            75026.0
Saved london_ensemble_forecast_jul_dec_2026.csv


In [5]:
# Get category-level actuals for July-Dec 2025 (naive seasonal baseline)
category_naive_raw = crime_full[crime_full["Date"].isin(naive_dates)]
category_naive_raw = category_naive_raw.groupby(["MajorCategory", "Date"])["CrimeCount"].sum().reset_index()
category_naive_raw["Month"] = category_naive_raw["Date"].dt.strftime("%Y-%m-%d").map(date_to_month2026)
category_naive = category_naive_raw[["MajorCategory", "Month", "CrimeCount"]].rename(
    columns={"MajorCategory": "Category", "CrimeCount": "Naive_Seasonal"}
)

print(category_naive.shape)
print(category_naive.head())

(78, 3)
                    Category    Month  Naive_Seasonal
0  ARSON AND CRIMINAL DAMAGE  2026-07            5227
1  ARSON AND CRIMINAL DAMAGE  2026-08            4856
2  ARSON AND CRIMINAL DAMAGE  2026-09            4507
3  ARSON AND CRIMINAL DAMAGE  2026-10            4653
4  ARSON AND CRIMINAL DAMAGE  2026-11            4447


In [8]:
# Reload the category forecast and reshape from wide to long
category_forecast_wide = pd.read_csv("category_full_forecast_aug_dec_2026.csv")
category_forecast_wide = category_forecast_wide.rename(columns={category_forecast_wide.columns[0]: "Category"})

category_forecast_long = category_forecast_wide.melt(id_vars="Category", var_name="Month", value_name="Forecast")

print(category_forecast_long.shape)
print(category_forecast_long.head())

(55, 3)
                               Category    Month  Forecast
0             ARSON AND CRIMINAL DAMAGE  2026-08      4877
1                              BURGLARY  2026-08      3966
2                         DRUG OFFENCES  2026-08      4105
3  MISCELLANEOUS CRIMES AGAINST SOCIETY  2026-08      1326
4                 POSSESSION OF WEAPONS  2026-08       796


In [9]:
# Merge SARIMA category forecast with naive seasonal baseline (inner join keeps only the 11 forecasted categories)
category_ensemble = category_forecast_long.merge(category_naive, on=["Category", "Month"])

# Apply the same 0.4/0.6 ensemble weights
category_ensemble["Ensemble_Forecast"] = (0.4 * category_ensemble["Forecast"]) + (0.6 * category_ensemble["Naive_Seasonal"])
category_ensemble["Ensemble_Forecast"] = category_ensemble["Ensemble_Forecast"].round(0)

print(category_ensemble.shape)
print(category_ensemble.head(10))

category_ensemble.to_csv("data/category_ensemble_forecast_aug_dec_2026.csv", index=False)
print("\nSaved category_ensemble_forecast_aug_dec_2026.csv")

(55, 5)
                               Category    Month  Forecast  Naive_Seasonal  \
0             ARSON AND CRIMINAL DAMAGE  2026-08      4877            4856   
1                              BURGLARY  2026-08      3966            3989   
2                         DRUG OFFENCES  2026-08      4105            4810   
3  MISCELLANEOUS CRIMES AGAINST SOCIETY  2026-08      1326            1098   
4                 POSSESSION OF WEAPONS  2026-08       796             616   
5                 PUBLIC ORDER OFFENCES  2026-08      7909            5004   
6                               ROBBERY  2026-08      2746            2745   
7                       SEXUAL OFFENCES  2026-08      2386            2372   
8                                 THEFT  2026-08     24444           24255   
9                      VEHICLE OFFENCES  2026-08      6962            7214   

   Ensemble_Forecast  
0             4864.0  
1             3980.0  
2             4528.0  
3             1189.0  
4              688

In [10]:
# Get borough x category actuals for July-Dec 2025 (naive seasonal baseline, full granularity)
bc_naive_raw = crime_full[crime_full["Date"].isin(naive_dates)]
bc_naive_raw = bc_naive_raw.groupby(["Borough", "MajorCategory", "Date"])["CrimeCount"].sum().reset_index()
bc_naive_raw["Month"] = bc_naive_raw["Date"].dt.strftime("%Y-%m-%d").map(date_to_month2026)
bc_naive = bc_naive_raw[["Borough", "MajorCategory", "Month", "CrimeCount"]].rename(
    columns={"MajorCategory": "Category", "CrimeCount": "Naive_Seasonal"}
)

print(bc_naive.shape)
print(bc_naive.head())

(2370, 4)
                Borough                   Category    Month  Naive_Seasonal
0  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE  2026-07             148
1  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE  2026-08             136
2  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE  2026-09             105
3  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE  2026-10             106
4  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE  2026-11             114


In [13]:
# Load your existing SARIMA borough x category forecast
bc_forecast = pd.read_csv("borough_category_forecast_aug_dec_2026.csv")
print(bc_forecast.columns.tolist())
print(bc_forecast.head())

['Borough', 'Category', '2026-08', '2026-09', '2026-10', '2026-11', '2026-12']
                Borough                              Category  2026-08  \
0  Barking and Dagenham             ARSON AND CRIMINAL DAMAGE      134   
1  Barking and Dagenham                              BURGLARY       67   
2  Barking and Dagenham                         DRUG OFFENCES       99   
3  Barking and Dagenham  MISCELLANEOUS CRIMES AGAINST SOCIETY       36   
4  Barking and Dagenham                 POSSESSION OF WEAPONS       15   

   2026-09  2026-10  2026-11  2026-12  
0      104      112      115      120  
1       72       79       91      101  
2      117      110      130      104  
3       36       38       36       37  
4       17       16       16       12  


In [14]:
# Reshape SARIMA borough x category forecast from wide to long
bc_forecast_long = bc_forecast.melt(id_vars=["Borough", "Category"], var_name="Month", value_name="Forecast")

# Merge with naive seasonal baseline
bc_ensemble = bc_forecast_long.merge(bc_naive, on=["Borough", "Category", "Month"])

# Apply the same 0.4/0.6 ensemble weights
bc_ensemble["Ensemble_Forecast"] = (0.4 * bc_ensemble["Forecast"]) + (0.6 * bc_ensemble["Naive_Seasonal"])
bc_ensemble["Ensemble_Forecast"] = bc_ensemble["Ensemble_Forecast"].round(0)

print(bc_ensemble.shape)
print(bc_ensemble.head(10))

bc_ensemble.to_csv("data/borough_category_ensemble_forecast_aug_dec_2026.csv", index=False)
print("\nSaved borough_category_ensemble_forecast_aug_dec_2026.csv")

(1760, 6)
                Borough                              Category    Month  \
0  Barking and Dagenham             ARSON AND CRIMINAL DAMAGE  2026-08   
1  Barking and Dagenham                              BURGLARY  2026-08   
2  Barking and Dagenham                         DRUG OFFENCES  2026-08   
3  Barking and Dagenham  MISCELLANEOUS CRIMES AGAINST SOCIETY  2026-08   
4  Barking and Dagenham                 POSSESSION OF WEAPONS  2026-08   
5  Barking and Dagenham                 PUBLIC ORDER OFFENCES  2026-08   
6  Barking and Dagenham                               ROBBERY  2026-08   
7  Barking and Dagenham                       SEXUAL OFFENCES  2026-08   
8  Barking and Dagenham                                 THEFT  2026-08   
9  Barking and Dagenham                      VEHICLE OFFENCES  2026-08   

   Forecast  Naive_Seasonal  Ensemble_Forecast  
0       134             136              135.0  
1        67              65               66.0  
2        99             142 

In [15]:
# Get per-borough actuals for March-June 2025 (naive seasonal baseline for Mar-Jun 2026)
mar_jun_2025_dates = pd.to_datetime(["2025-03-01", "2025-04-01", "2025-05-01", "2025-06-01"])
mar_jun_2026_labels = ["2026-03", "2026-04", "2026-05", "2026-06"]
date_to_label_mj = dict(zip(mar_jun_2025_dates, mar_jun_2026_labels))

borough_naive_mj_raw = crime_full[crime_full["Date"].isin(mar_jun_2025_dates)]
borough_naive_mj_raw = borough_naive_mj_raw.groupby(["Borough", "Date"])["CrimeCount"].sum().reset_index()
borough_naive_mj_raw["Month"] = borough_naive_mj_raw["Date"].map(date_to_label_mj)
borough_naive_mj = borough_naive_mj_raw[["Borough", "Month", "CrimeCount"]].rename(columns={"CrimeCount": "Naive_Seasonal"})

print(borough_naive_mj.shape)
print(borough_naive_mj.head())

(128, 3)
                Borough    Month  Naive_Seasonal
0  Barking and Dagenham  2026-03            1752
1  Barking and Dagenham  2026-04            1786
2  Barking and Dagenham  2026-05            1718
3  Barking and Dagenham  2026-06            1773
4                Barnet  2026-03            2435


In [16]:
# Load existing March-June SARIMA validation (has Borough, Month, Actual, Forecast, Pct_Error)
mar_jun_val = pd.read_csv("data/borough_actual_vs_forecast_mar_jun_2026.csv")
print(mar_jun_val.columns.tolist())
print(mar_jun_val.shape)
print(mar_jun_val.head())

['Borough', 'Month', 'Actual', 'Forecast', 'Pct_Error']
(128, 5)
                Borough    Month  Actual  Forecast  Pct_Error
0  Barking and Dagenham  2026-03    1761      1699        3.5
1  Barking and Dagenham  2026-04    1650      1625        1.5
2  Barking and Dagenham  2026-05    1830      1658        9.4
3  Barking and Dagenham  2026-06    1870      1640       12.3
4                Barnet  2026-03    2651      2543        4.1


In [19]:
# Rebuild July's SARIMA forecast (from your saved forecast file)
jul_forecast_full = pd.read_csv("borough_future_forecasts_Jul-Dec_2026.csv")
jul_forecast_only = jul_forecast_full[jul_forecast_full["Month"] == "2026-07"].copy()

# Rebuild July's real actuals (from the updated MPS file with July data)
newer = pd.read_csv("MPS_Borough_Level_Crime _Most_Recent_24_months_new.csv")
jul_col = "202607"
borough_actual_jul = newer.groupby("BOCU")[jul_col].sum().reset_index()
borough_actual_jul.columns = ["Borough", "Actual_2026-07"]
borough_actual_jul = borough_actual_jul[~borough_actual_jul["Borough"].isin(["Unknown", "Aviation Policing"])]

print(jul_forecast_only.shape)
print(borough_actual_jul.shape)

KeyError: 'Column not found: 202607'

In [20]:
import os
print([f for f in os.listdir() if "Recent_24" in f])

['MPS_Borough_Level_Crime_Recent_24_month.csv', 'MPS_Borough_Level_Crime _Most_Recent_24_months_new.csv']


In [21]:
newer_check = pd.read_csv("MPS_Borough_Level_Crime _Most_Recent_24_months_new.csv")
print(newer_check.columns.tolist()[-8:])

['202511', '202512', '202601', '202602', '202603', '202604', '202605', '202606']


In [22]:
import os
print(os.listdir())

['.bashrc', '.profile', '.local', '.ipython', '.jupyter', '.npm', '.ipynb_checkpoints', '.cache', '.config', 'London Crime analysis(FF).ipynb', 'London Crime analysis(VF).ipynb', 'London Crime analysis(EBT).ipynb', 'borough_future_forecasts_Jul-Dec_2026.csv', 'crime type forcast (LCA).ipynb', 'sarima_autotuning.ipynb', 'category_trend_ranking.png', 'dashboardstuff.ipynb', 'each borugh actual vs forcast for dashboard .ipynb', 'London_Borough_Claimant_Count_WIDE.csv', 'test july validation.ipynb', 'checking forcast usinf ensemble version.ipynb', 'Dessertation ', 'ensemble_future_forecast.ipynb', 'borough_category_forecast_aug_dec_2026.csv', 'MPS_Borough_Level_Crime_Historical.csv', 'MPS_Borough_Level_Crime_Recent_24_month.csv', 'data', 'London Crime analysis[EDA].ipynb', 'London Crime analysis[DC].ipynb', 'London Crime analysis[DV].ipynb', 'figures', 'London Crime Analysis[SA].ipynb', 'London Crime analysis(ML).ipynb', 'London Crime analysis(AM).ipynb', 'London Crime analysis[K means].ip

In [23]:
newer_july = pd.read_csv("MPS_Borough_Level_Crime_Most recent_July_month_validation.csv")
print(newer_july.columns.tolist())
print(newer_july.head())

['Group', 'SubGroup', 'BOCU', '202408', '202409', '202410', '202411', '202412', '202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604', '202605', '202606', '202607']
                                  Group                         SubGroup  \
0  MISCELLANEOUS CRIMES AGAINST SOCIETY      MISC CRIMES AGAINST SOCIETY   
1             ARSON AND CRIMINAL DAMAGE                            ARSON   
2             ARSON AND CRIMINAL DAMAGE                  CRIMINAL DAMAGE   
3                              BURGLARY  BURGLARY BUSINESS AND COMMUNITY   
4                              BURGLARY           RES BURGLARY OF A HOME   

                   BOCU  202408  202409  202410  202411  202412  202501  \
0     Aviation Policing       0       0       0       0       0       0   
1  Barking and Dagenham      10       9       5       7       9       7   
2  Barking and Dagenham     114      80     103 

In [24]:
jul_col = "202607"
borough_actual_jul = newer_july.groupby("BOCU")[jul_col].sum().reset_index()
borough_actual_jul.columns = ["Borough", "Actual_2026-07"]
borough_actual_jul = borough_actual_jul[~borough_actual_jul["Borough"].isin(["Unknown", "Aviation Policing"])]

print(borough_actual_jul.shape)
print(borough_actual_jul.head())

(32, 2)
                Borough  Actual_2026-07
1  Barking and Dagenham            1874
2                Barnet            2493
3                Bexley            1538
4                 Brent            2935
5               Bromley            2278


In [25]:
july_full = jul_forecast_only[["Borough", "Month", "Forecast"]].merge(borough_actual_jul, on="Borough")
july_full = july_full.rename(columns={"Actual_2026-07": "Actual"})
july_full = july_full.merge(borough_naive[borough_naive["Month"] == "2026-07"][["Borough", "Naive_Seasonal"]], on="Borough")

july_full["Pct_Error"] = (abs(july_full["Actual"] - july_full["Forecast"]) / july_full["Actual"] * 100).round(1)
july_full["Ensemble_Forecast"] = (0.4 * july_full["Forecast"]) + (0.6 * july_full["Naive_Seasonal"])
july_full["Ensemble_Forecast"] = july_full["Ensemble_Forecast"].round(0)
july_full["Ensemble_Pct_Error"] = (abs(july_full["Actual"] - july_full["Ensemble_Forecast"]) / july_full["Actual"] * 100).round(1)

# Combine March-June + July into one file
full_validation = pd.concat([mar_jun_full, july_full], ignore_index=True)
full_validation = full_validation.sort_values(["Borough", "Month"]).reset_index(drop=True)

print(full_validation.shape)
print(full_validation.head(10))

full_validation.to_csv("data/borough_validation_mar_jul_2026_with_ensemble.csv", index=False)
print("\nSaved borough_validation_mar_jul_2026_with_ensemble.csv")

(160, 8)
                Borough    Month  Actual  Forecast  Pct_Error  Naive_Seasonal  \
0  Barking and Dagenham  2026-03    1761      1699        3.5            1752   
1  Barking and Dagenham  2026-04    1650      1625        1.5            1786   
2  Barking and Dagenham  2026-05    1830      1658        9.4            1718   
3  Barking and Dagenham  2026-06    1870      1640       12.3            1773   
4  Barking and Dagenham  2026-07    1874      1930        3.0            1792   
5                Barnet  2026-03    2651      2543        4.1            2435   
6                Barnet  2026-04    2352      2466        4.8            2284   
7                Barnet  2026-05    2519      2626        4.2            2440   
8                Barnet  2026-06    2590      2533        2.2            2353   
9                Barnet  2026-07    2493      2807       12.6            2587   

   Ensemble_Forecast  Ensemble_Pct_Error  
0             1731.0                 1.7  
1            

In [26]:
full_validation["Naive_Pct_Error"] = (abs(full_validation["Actual"] - full_validation["Naive_Seasonal"]) / full_validation["Actual"] * 100).round(1)

full_validation.to_csv("data/borough_validation_mar_jul_2026_with_ensemble.csv", index=False)
print("Saved with Naive Seasonal error included!")
print(full_validation.head())

Saved with Naive Seasonal error included!
                Borough    Month  Actual  Forecast  Pct_Error  Naive_Seasonal  \
0  Barking and Dagenham  2026-03    1761      1699        3.5            1752   
1  Barking and Dagenham  2026-04    1650      1625        1.5            1786   
2  Barking and Dagenham  2026-05    1830      1658        9.4            1718   
3  Barking and Dagenham  2026-06    1870      1640       12.3            1773   
4  Barking and Dagenham  2026-07    1874      1930        3.0            1792   

   Ensemble_Forecast  Ensemble_Pct_Error  Naive_Pct_Error  
0             1731.0                 1.7              0.5  
1             1722.0                 4.4              8.2  
2             1694.0                 7.4              6.1  
3             1720.0                 8.0              5.2  
4             1847.0                 1.4              4.4  


In [27]:
check = pd.read_csv("data/borough_validation_mar_jul_2026_with_ensemble.csv")
print(check.columns.tolist())

['Borough', 'Month', 'Actual', 'Forecast', 'Pct_Error', 'Naive_Seasonal', 'Ensemble_Forecast', 'Ensemble_Pct_Error', 'Naive_Pct_Error']
